Author: Rishabh Srivastava

Note: I kept explanations short to avoid unnecessary length (as requested). Timestamp: 2026-06-14

# Assignment 2 — Sentiment Analysis and Topic Modelling

## Part A — Sentiment Analysis

### Q1. Load and Preprocess

Load sentiment_sentences.csv using pandas. Preprocess the text column — convert to lower case, remove punctuation and special characters, tokenize, and remove stop words. Print the cleaned tokens for one sample review.

Note: I first clean text to remove noise so models learn from real words, not punctuation or filler.
This keeps explanations short: preprocessing lowercases, strips punctuation, tokenizes, and removes common stop words so the model focuses on meaningful tokens.

In [ ]:
import pandas as pd
import numpy as np
import re
import string
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download NLTK resources (only necessary corpora)
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /Users/krishna/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/krishna/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     /Users/krishna/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [15]:
# Load the dataset
df = pd.read_csv('sentiment_sentences.csv')
print(f"Dataset shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())

Dataset shape: (3000, 3)

Columns: ['text', 'sentiment', 'source']

First few rows:
                                                text  sentiment  source
0  So there is no way for me to plug it in here i...          0  amazon
1                        Good case, Excellent value.          1  amazon
2                             Great for the jawbone.          1  amazon
3  Tied to charger for conversations lasting more...          0  amazon
4                                  The mic is great.          1  amazon


In [16]:
# Get English stop words
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    """
    Preprocess text: lowercase, remove punctuation/special characters, tokenize, remove stopwords
    """
    # Convert to lowercase
    text = text.lower()
    
    # Remove punctuation and special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    
    # Tokenize
    tokens = word_tokenize(text)
    
    # Remove stop words
    tokens = [token for token in tokens if token not in stop_words]
    
    return tokens

# Apply preprocessing to all text
df['tokens'] = df['text'].apply(preprocess_text)

# Print cleaned tokens for one sample review
sample_index = 0
print(f"Original text: {df.iloc[sample_index]['text']}")
print(f"Cleaned tokens: {df.iloc[sample_index]['tokens']}")

Original text: So there is no way for me to plug it in here in the US unless I go by a converter.
Cleaned tokens: ['way', 'plug', 'us', 'unless', 'go', 'converter']


### Q2. Naive Bayes from scratch + Evaluation

Split the data into training (80%) and test (20%) sets using a stratified split. Implement the Multinomial Naive Bayes classifier from scratch — compute the class priors P(c) = N_c / N and the word likelihoods with Laplace (add-1) smoothing: P(w|c) = (count(w,c) + 1) / (sum of counts in c + |V|). Predict the sentiment of the test set and report the Accuracy and the F1-score.

**Implementation approach:**

Multinomial Naive Bayes works by applying Bayes theorem with the naive assumption that words are conditionally independent given the class.

- **Class priors:**  — fraction of training documents belonging to class c.
- **Word likelihoods with Laplace (add-1) smoothing:**  — smoothing ensures unseen words do not zero-out the probability.
- **Prediction:** For each test document, compute  for each class and pick the class with the highest score. Log-probabilities are used to avoid numerical underflow from multiplying many small probabilities.

The classifier is implemented entirely from scratch without any sklearn classifier.

In [17]:
# Split data into training (80%) and test (20%) with stratified split
X_train, X_test, y_train, y_test = train_test_split(
    df['tokens'].tolist(), 
    df['sentiment'].tolist(), 
    test_size=0.2, 
    random_state=42,
    stratify=df['sentiment']
)

print(f"Training set size: {len(X_train)}")
print(f"Test set size: {len(X_test)}")
print(f"Training sentiment distribution: {Counter(y_train)}")
print(f"Test sentiment distribution: {Counter(y_test)}")

Training set size: 2400
Test set size: 600
Training sentiment distribution: Counter({1: 1200, 0: 1200})
Test sentiment distribution: Counter({1: 300, 0: 300})


In [18]:
class MultinomialNaiveBayes:
    """
    Multinomial Naive Bayes classifier implemented from scratch with Laplace smoothing
    """
    
    def __init__(self):
        self.class_priors = {}
        self.word_likelihoods = {}
        self.vocabulary = set()
        self.classes = None
    
    def fit(self, X_train, y_train):
        """
        Train the classifier
        
        Parameters:
        X_train: list of token lists
        y_train: list of class labels
        """
        self.classes = list(set(y_train))
        n_samples = len(y_train)
        
        # Build vocabulary from all training documents
        for tokens in X_train:
            self.vocabulary.update(tokens)
        self.vocabulary = sorted(list(self.vocabulary))
        vocab_size = len(self.vocabulary)
        
        print(f"Vocabulary size: {vocab_size}")
        
        # Compute class priors P(c) = N_c / N
        for c in self.classes:
            self.class_priors[c] = y_train.count(c) / n_samples
        
        print(f"Class priors: {self.class_priors}")
        
        # Compute word likelihoods with Laplace smoothing
        # P(w|c) = (count(w,c) + 1) / (sum of counts in c + |V|)
        for c in self.classes:
            # Get all documents belonging to class c
            class_documents = [X_train[i] for i in range(len(X_train)) if y_train[i] == c]
            
            # Count word occurrences in class c
            word_counts = Counter()
            total_words_in_class = 0
            
            for tokens in class_documents:
                word_counts.update(tokens)
                total_words_in_class += len(tokens)
            
            # Compute likelihoods with Laplace smoothing
            self.word_likelihoods[c] = {}
            for word in self.vocabulary:
                count_wc = word_counts.get(word, 0)
                likelihood = (count_wc + 1) / (total_words_in_class + vocab_size)
                self.word_likelihoods[c][word] = likelihood
    
    def predict(self, X_test):
        """
        Predict class labels for test documents
        
        Parameters:
        X_test: list of token lists
        
        Returns:
        predictions: list of predicted class labels
        """
        predictions = []
        
        for tokens in X_test:
            best_class = None
            best_log_prob = float('-inf')
            
            for c in self.classes:
                # Start with log of class prior
                log_prob = np.log(self.class_priors[c])
                
                # Add log of word likelihoods
                for word in tokens:
                    if word in self.word_likelihoods[c]:
                        log_prob += np.log(self.word_likelihoods[c][word])
                    # If word not in vocabulary, it contributes 0 (smoothed)
                
                if log_prob > best_log_prob:
                    best_log_prob = log_prob
                    best_class = c
            
            predictions.append(best_class)
        
        return predictions

In [19]:
# Train the Naive Bayes classifier
nb = MultinomialNaiveBayes()
nb.fit(X_train, y_train)

Vocabulary size: 4522
Class priors: {0: 0.5, 1: 0.5}


In [20]:
# Predict on test set
y_pred = nb.predict(X_test)

# Calculate accuracy and F1-score
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score: {f1:.4f}")

Accuracy: 0.8150
F1-Score: 0.8165


## Part B — Topic Modelling with LDA

### Q3. Document-Term Matrix, LDA and Topic Words

Build the document-term (word-count) matrix from the review sentences after removing English stop words (you may use CountVectorizer). Apply the LDA algorithm to create K = 3 topics, then list the top 10 highest-probability words in each topic.

**Notes**

Latent Dirichlet Allocation (LDA) is an unsupervised generative model for topic modelling. It assumes:
- Each **topic** is a probability distribution over the vocabulary (words likely to co-occur are grouped together).
- Each **document** is a mixture of topics — i.e., it has a probability distribution over K topics.

Here we use K=3 topics and build the document-term matrix with  (word counts, English stop words removed). LDA is trained on this matrix without using the sentiment or source labels — it discovers structure purely from word co-occurrence patterns.

In [21]:
# Build document-term matrix using CountVectorizer with stop words removal
vectorizer = CountVectorizer(stop_words='english')
dtm = vectorizer.fit_transform(df['text'])

print(f"Document-Term Matrix shape: {dtm.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

Document-Term Matrix shape: (3000, 4896)
Vocabulary size: 4896


In [22]:
# Apply LDA with K=3 topics
lda = LatentDirichletAllocation(n_components=3, random_state=42, max_iter=10)
lda.fit(dtm)

print(f"LDA model fitted with 3 topics")

LDA model fitted with 3 topics


In [23]:
# Get feature names (words)
feature_names = vectorizer.get_feature_names_out()

# Print top 10 words for each topic
def print_top_words(model, feature_names, n_top_words=10):
    for topic_idx, topic in enumerate(model.components_):
        message = f"Topic #{topic_idx}: "
        message += " ".join([feature_names[i] for i in topic.argsort()[:-n_top_words - 1:-1]])
        print(message)
    print()

print("Top 10 words in each topic:")
print_top_words(lda, feature_names, n_top_words=10)

Top 10 words in each topic:
Topic #0: phone film just like battery love does disappointed ve terrible
Topic #1: great good food service place bad just quality time works
Topic #2: movie film don time really recommend place good sound best



### Q4. Document-Topic Distribution

For at least 5 documents, print the distribution over the 3 topics and the dominant (most probable) topic. Optionally, compare the dominant topic against the source column to see how the discovered topics relate to the three review domains.

**Interpreting document-topic distributions:**

 returns a matrix of shape (n_documents, K) where each row sums to 1. Each value represents how strongly a document is associated with a given topic.

The **dominant topic** is simply  of the distribution — the topic with the highest probability for that document.

Optionally, comparing dominant topics against the  column (amazon / yelp / imdb) lets us check whether LDA topics naturally align with review domains, even though source labels were never given to LDA during training.

In [24]:
# Get document-topic distributions
doc_topic_distributions = lda.transform(dtm)

print(f"Document-topic distribution shape: {doc_topic_distributions.shape}")

Document-topic distribution shape: (3000, 3)


In [25]:
# Print distribution and dominant topic for 5 sample documents
num_samples = 5
sample_indices = list(range(num_samples))

for idx in sample_indices:
    print(f"\nDocument #{idx}:")
    print(f"Text: {df.iloc[idx]['text']}")
    print(f"Source: {df.iloc[idx]['source']}")
    print(f"Sentiment: {df.iloc[idx]['sentiment']}")
    print(f"Topic distribution: {doc_topic_distributions[idx]}")
    dominant_topic = np.argmax(doc_topic_distributions[idx])
    print(f"Dominant topic: Topic #{dominant_topic}")


Document #0:
Text: So there is no way for me to plug it in here in the US unless I go by a converter.
Source: amazon
Sentiment: 0
Topic distribution: [0.0684345  0.84517423 0.08639128]
Dominant topic: Topic #1

Document #1:
Text: Good case, Excellent value.
Source: amazon
Sentiment: 1
Topic distribution: [0.07425894 0.85628089 0.06946017]
Dominant topic: Topic #1

Document #2:
Text: Great for the jawbone.
Source: amazon
Sentiment: 1
Topic distribution: [0.11127743 0.7768542  0.11186836]
Dominant topic: Topic #1

Document #3:
Text: Tied to charger for conversations lasting more than 45 minutes.MAJOR PROBLEMS!!
Source: amazon
Sentiment: 0
Topic distribution: [0.68792613 0.15635562 0.15571824]
Dominant topic: Topic #0

Document #4:
Text: The mic is great.
Source: amazon
Sentiment: 1
Topic distribution: [0.11140114 0.76590538 0.12269348]
Dominant topic: Topic #1


In [26]:
# Optional: Compare dominant topics with source column
# Get dominant topic for all documents
df['dominant_topic'] = np.argmax(doc_topic_distributions, axis=1)

# Cross-tabulation of source vs dominant topic
print("\nCross-tabulation of Source vs Dominant Topic:")
print(pd.crosstab(df['source'], df['dominant_topic']))


Cross-tabulation of Source vs Dominant Topic:
dominant_topic    0    1    2
source                       
amazon          323  373  304
imdb            289  248  463
yelp            230  498  272


In [27]:
# Calculate percentage distribution of topics for each source
print("\nTopic distribution percentage by source:")
for source in df['source'].unique():
    source_df = df[df['source'] == source]
    topic_counts = source_df['dominant_topic'].value_counts(normalize=True).sort_index()
    print(f"\n{source}:")
    for topic, pct in topic_counts.items():
        print(f"  Topic {topic}: {pct:.2%}")


Topic distribution percentage by source:

amazon:
  Topic 0: 32.30%
  Topic 1: 37.30%
  Topic 2: 30.40%

yelp:
  Topic 0: 23.00%
  Topic 1: 49.80%
  Topic 2: 27.20%

imdb:
  Topic 0: 28.90%
  Topic 1: 24.80%
  Topic 2: 46.30%
